# GEO · E0 + E1 — first ML-in-the-loop flight (agi-semantic-core Phase 8)

**What this does**
- **E0 — layer sweep:** does a modern 1.5B LLM (Qwen2.5-1.5B) carry more of the dictionary's grounded 14D geometry than the 22M embedder measured in Phase 6 (linear R² 0.514, 95.5% complement compression)? Probes every hidden layer, and re-runs the MiniLM anchor in-notebook for parity.
- **E1 — angular instillation (the flagship):** fine-tunes `all-MiniLM-L6-v2` against the dictionary's angular contract (targets = true grounded angles; complements → ~90°), with **held-out pairs** (the model must learn the *rule*, not the list), a retention channel (STS-B teacher self-distillation), three mixing variants, and a **pre-registered verdict**. Generalization probes: held-out relations, bare-name forms, and unseen-antonym twins.

**How to run:** Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save · then Runtime ▸ **Run all**. ≈60–100 min total; free tier is fine. If the runtime dies mid-way, Run all again — finished stages reload from cache while the VM lives.

**Output:** `geo_e0e1_results.zip` (metrics, plots, verdict, best tuned model). A browser download fires at the end; an optional save-to-Drive cell is last.

*Staged 2026-08-20 · executes Phase 6 §7 path-forward item 3 ("angular loss training") · design doc: `~/qualia-algebra/internal/GEOMETRIC-REASONING-PASS.md`*

In [ ]:
# ── Setup: GPU check, installs, repo clone ────────────────────────────────────
import subprocess, sys
from pathlib import Path

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')
import torch
assert torch.cuda.is_available(), (
    'No GPU. Colab menu: Runtime > Change runtime type > Hardware accelerator: T4 GPU, then Run all again.')

print('Installing packages (~1-2 min)...')
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'sentence-transformers>=3.0','transformers','accelerate','datasets','scikit-learn','scipy','pandas'],
    check=True)

REPO = Path('/content/agi-semantic-core')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/QAv2/agi-semantic-core.git', str(REPO)], check=True)
DB = REPO/'db'/'semantic.db'
assert DB.exists(), 'semantic.db missing from clone'

RESULTS = Path('/content/geo_results'); RESULTS.mkdir(exist_ok=True)
import sqlite3
n_concepts = sqlite3.connect(DB).execute('SELECT COUNT(*) FROM concepts').fetchone()[0]
print(f'Dictionary loaded: {n_concepts} concepts (expected 3033)')
SEED = 42

In [ ]:
# ── Dictionary extraction: 14D vectors, relations with true angles, splits ────
import numpy as np, sqlite3
from collections import Counter
rng = np.random.default_rng(SEED)

con = sqlite3.connect(DB); con.row_factory = sqlite3.Row
DIMS = ['x','y','z','e','f','g','h','fx','fy','fz','fe','ff','fg','fh']   # 14D, both w slots excluded
rows = con.execute(f"SELECT id,name,description,{','.join(DIMS)} FROM concepts ORDER BY id").fetchall()
names = [r['name'] for r in rows]
texts = [f"{r['name']}: {r['description']}" for r in rows]                 # Phase 6 input format
Y14   = np.array([[r[d] for d in DIMS] for r in rows], dtype=np.float64)
Y7    = Y14[:, :7]                                                          # essence 7D = angle_8d space
id2idx = {r['id']: i for i, r in enumerate(rows)}
name_set  = {r['name'].upper() for r in rows}
alias_set = {a['alias'].upper() for a in con.execute('SELECT alias FROM aliases')}

pairs = []
for r in con.execute("SELECT concept1_id c1, concept2_id c2, rel_type, angle_4d, angle_8d FROM relations"):
    if r['c1'] not in id2idx or r['c2'] not in id2idx: continue
    ta = r['angle_8d'] if r['angle_8d'] else r['angle_4d']                  # Phase 6 convention
    if not ta: continue
    pairs.append((id2idx[r['c1']], id2idx[r['c2']], r['rel_type'], float(ta)))
print(f'{len(pairs)} usable relations —', dict(Counter(p[2] for p in pairs)))

# Stratified 80/20 split by rel_type — held-out pairs are NEVER trained on
by_type = {}
for p in pairs: by_type.setdefault(p[2], []).append(p)
train_rel, test_rel = [], []
for t, ps in sorted(by_type.items()):
    idx = rng.permutation(len(ps)); cut = int(0.8*len(ps))
    train_rel += [ps[i] for i in idx[:cut]]; test_rel += [ps[i] for i in idx[cut:]]
print(f'relation split: {len(train_rel)} train / {len(test_rel)} held out')

# Random-pair channel: unrelated pairs with true 7D grounded angles (global metric supervision)
def angle7(i, j):
    a, b = Y7[i], Y7[j]
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-9 or nb < 1e-9: return None
    return float(np.degrees(np.arccos(np.clip(a@b/(na*nb), -1, 1))))
related = {(min(p[0],p[1]), max(p[0],p[1])) for p in pairs}
rand_pairs, seen = [], set()
while len(rand_pairs) < 12000:
    i, j = (int(v) for v in rng.integers(0, len(names), 2))
    key = (min(i,j), max(i,j))
    if i == j or key in related or key in seen: continue
    a = angle7(i, j)
    if a is None: continue
    seen.add(key); rand_pairs.append((i, j, 'random', a))
rand_train, rand_test = rand_pairs[:10000], rand_pairs[10000:]
print(f'random-pair channel: {len(rand_train)} train / {len(rand_test)} held out')

## E0 — layer sweep: does scale already carry the geometry?
Forward-pass all concepts through **Qwen2.5-1.5B**, mean-pool every layer, fit linear probes to the 14D grounded space, and measure complement compression per layer — against the MiniLM anchor run with identical code. Parity track = full-sample OLS, no intercept (exactly Phase 6). Honest track = 80/20 ridge with alpha CV.

In [ ]:
# ── E0a: per-layer hidden states from Qwen2.5-1.5B (cached) ──────────────────
import numpy as np, torch
HS_PATH = RESULTS/'qwen_hidden.npy'
if HS_PATH.exists():
    H = np.load(HS_PATH); print('loaded cached hidden states', H.shape)
else:
    from transformers import AutoTokenizer, AutoModel
    QWEN = 'Qwen/Qwen2.5-1.5B'
    tok = AutoTokenizer.from_pretrained(QWEN)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModel.from_pretrained(QWEN, torch_dtype=torch.float16).cuda().eval()
    chunks, B = [], 16
    with torch.no_grad():
        for s in range(0, len(texts), B):
            enc = tok(texts[s:s+B], return_tensors='pt', padding=True,
                      truncation=True, max_length=64).to('cuda')
            out = model(**enc, output_hidden_states=True)
            mask = enc['attention_mask'].unsqueeze(-1).to(torch.float16)
            layers = [((hs*mask).sum(1)/mask.sum(1)).float().cpu().numpy()
                      for hs in out.hidden_states]
            chunks.append(np.stack(layers, 1))                       # [B, L+1, D]
            if (s//B) % 25 == 0: print(f'  {s}/{len(texts)}')
    H = np.concatenate(chunks, 0)
    np.save(HS_PATH, H)
    del model; torch.cuda.empty_cache()
print('hidden states:', H.shape, '(concepts × layers × dim)')

In [ ]:
# ── E0b: MiniLM anchor embeddings (Phase 6 parity baseline) ──────────────────
import numpy as np
from sentence_transformers import SentenceTransformer
MINI_PATH = RESULTS/'minilm_base.npy'
if MINI_PATH.exists():
    E_mini = np.load(MINI_PATH)
else:
    st_base = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
    E_mini = st_base.encode(texts, batch_size=256, show_progress_bar=False, convert_to_numpy=True)
    np.save(MINI_PATH, E_mini)
    del st_base
print('MiniLM anchor:', E_mini.shape)

In [ ]:
# ── E0c: probes per layer — R² (parity + held-out) and complement compression ─
import numpy as np, json, pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split

comp_all = [p for p in pairs if p[2] == 'complement']

def probe_space(X):
    # Parity: full-sample OLS, no intercept — exactly Phase 6 (reported R²=0.5144)
    W, *_ = np.linalg.lstsq(X, Y14, rcond=None)
    P = X @ W
    r2_par = 1 - ((Y14-P)**2).sum() / ((Y14-Y14.mean(0))**2).sum()
    # Honest: 80/20 ridge with alpha CV
    tr, te = train_test_split(np.arange(len(X)), test_size=0.2, random_state=SEED)
    ridge = RidgeCV(alphas=[1e-3,1e-2,1e-1,1,10,100]).fit(X[tr], Y14[tr])
    r2_gen = ridge.score(X[te], Y14[te])
    # Complement compression on parity predictions (14D angles — Phase 6 convention)
    errs, comp = [], 0
    for i,j,_,ta in comp_all:
        a, b = P[i], P[j]
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na < 1e-10 or nb < 1e-10: continue
        pa = float(np.degrees(np.arccos(np.clip(a@b/(na*nb), -1, 1))))
        errs.append(pa-ta); comp += (pa < ta)
    errs = np.array(errs)
    return dict(r2_parity=float(r2_par), r2_heldout=float(r2_gen),
                comp_mean_shift=float(errs.mean()),
                comp_pct_compressed=float(100*comp/len(errs)))

mini_res = probe_space(E_mini.astype(np.float64))
print('MiniLM-384 anchor:', {k: round(v,3) for k,v in mini_res.items()},
      '  [Phase 6 at 2,237 concepts: R²=0.514, 95.5% compressed]')
sweep = []
for L in range(H.shape[1]):
    r = probe_space(H[:, L, :].astype(np.float64))
    sweep.append({'layer': L, **r})
    print(f"layer {L:2d}  parityR2={r['r2_parity']:.3f}  heldoutR2={r['r2_heldout']:.3f}  "
          f"comp_shift={r['comp_mean_shift']:+.1f}deg  compressed={r['comp_pct_compressed']:.0f}%")
pd.DataFrame(sweep).to_csv(RESULTS/'e0_layer_sweep.csv', index=False)
json.dump({'minilm_anchor': mini_res, 'layers': sweep},
          open(RESULTS/'e0_results.json','w'), indent=1)

In [ ]:
# ── E0d: plot + E0 verdict ────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, pandas as pd
df = pd.read_csv(RESULTS/'e0_layer_sweep.csv')
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(df.layer, df.r2_parity, 'o-', label='parity R² (OLS, full-sample)')
ax[0].plot(df.layer, df.r2_heldout, 's-', label='held-out R² (ridge)')
ax[0].axhline(mini_res['r2_parity'], ls='--', c='gray', label='MiniLM anchor (parity)')
ax[0].set_xlabel('Qwen2.5-1.5B layer'); ax[0].set_ylabel('R² → 14D grounded space'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(df.layer, df.comp_pct_compressed, 'o-', color='firebrick', label='% complement pairs compressed')
ax[1].axhline(mini_res['comp_pct_compressed'], ls='--', c='gray', label='MiniLM anchor')
ax[1].set_xlabel('layer'); ax[1].set_ylabel('% compressed (target: lower)'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(RESULTS/'e0_plot.png', dpi=140); plt.show()

best = df.loc[df.r2_heldout.idxmax()]
delta = best.r2_heldout - mini_res['r2_heldout']
print(f"\nE0 VERDICT: best layer {int(best.layer)} held-out R2={best.r2_heldout:.3f} "
      f"vs MiniLM {mini_res['r2_heldout']:.3f} (delta {delta:+.3f})")
print('  ->', 'scale/depth DOES carry more of the grounded geometry' if delta > 0.05 else
      'the gap PERSISTS at 1.5B — grounded geometry is not an artifact of embedder size')
print(f"  complement compression at best layer: {best.comp_pct_compressed:.0f}% "
      f"(MiniLM {mini_res['comp_pct_compressed']:.0f}%)")

## E1 — angular instillation: can the contract be TRAINED IN?
Fine-tune MiniLM with `CosineSimilarityLoss` (= MSE between predicted cosine and target cosine) on three channels: dictionary relations (targets = true grounded angles), random concept pairs (global metric), and STS-B pairs with **frozen-teacher cosines** (retention). Three mixing variants sweep the instillation/retention trade-off. Everything below is judged on **held-out** pairs.

In [ ]:
# ── E1a: training datasets (3 variants) + frozen-teacher distillation ─────────
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

def cosd(deg): return float(np.cos(np.radians(deg)))
def pair_rows(plist, reps=1):
    rws = [{'sentence1': texts[i], 'sentence2': texts[j], 'score': cosd(ta)}
           for i, j, t, ta in plist]
    return rws * reps

stsb = load_dataset('sentence-transformers/stsb')
sts_train, sts_dev = stsb['train'], stsb['validation']
teacher = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
ta_ = teacher.encode(list(sts_train['sentence1']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
tb_ = teacher.encode(list(sts_train['sentence2']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
t_cos = (ta_ * tb_).sum(1)
distill_rows = [{'sentence1': s1, 'sentence2': s2, 'score': float(c)}
                for s1, s2, c in zip(sts_train['sentence1'], sts_train['sentence2'], t_cos)]
del ta_, tb_

dict_rows = pair_rows(train_rel, reps=2) + pair_rows(rand_train)
VARIANTS = {
    'V0_pure':         dict_rows,                          # max instillation, no retention channel
    'V1_balanced':     dict_rows + distill_rows,           # ~1:0.35 dict:distill
    'V2_conservative': dict_rows + distill_rows*4,         # retention-heavy
}
for k, v in VARIANTS.items(): print(f'{k}: {len(v)} examples')

In [ ]:
# ── E1b: fine-tune per variant (skips any variant already trained) ────────────
import numpy as np, torch, random, shutil
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
MODELS = RESULTS/'models'; MODELS.mkdir(exist_ok=True)
for vname, rows_v in VARIANTS.items():
    outdir = MODELS/vname
    if (outdir/'config.json').exists():
        print(vname, 'already trained — skip'); continue
    print(f'=== training {vname} ({len(rows_v)} examples) ===')
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
    ds = Dataset.from_list(rows_v).shuffle(seed=SEED)
    args = SentenceTransformerTrainingArguments(
        output_dir=f'/content/tmp_{vname}', num_train_epochs=4,
        per_device_train_batch_size=64, learning_rate=2e-5, warmup_ratio=0.1,
        fp16=True, logging_steps=250, save_strategy='no', report_to='none', seed=SEED)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=ds,
                               loss=losses.CosineSimilarityLoss(model)).train()
    model.save(str(outdir))
    del model; torch.cuda.empty_cache()
    shutil.rmtree(f'/content/tmp_{vname}', ignore_errors=True)
print('E1 training done')

In [ ]:
# ── E1c: evaluation battery — held-out geometry, retention, probe, twins ──────
import numpy as np, json, torch, pandas as pd
from scipy.stats import spearmanr, pearsonr
from sentence_transformers import SentenceTransformer

def pred_angles(model, plist, form):
    strs = texts if form == 'desc' else [n.lower() for n in names]
    idxs = sorted({i for p in plist for i in p[:2]})
    sub = model.encode([strs[i] for i in idxs], batch_size=256,
                       convert_to_numpy=True, normalize_embeddings=True)
    pos = {ix: k for k, ix in enumerate(idxs)}
    return [(t, ta, float(np.degrees(np.arccos(np.clip(sub[pos[i]] @ sub[pos[j]], -1, 1)))))
            for i, j, t, ta in plist]

def eval_model(model, tag):
    res = {'tag': tag}
    for form in ['desc', 'name']:
        ang = pred_angles(model, test_rel, form)
        for t in ['complement', 'synonym', 'affinity']:
            sel = [(ta, pa) for (tt, ta, pa) in ang if tt == t]
            if not sel: continue
            ta_, pa_ = map(np.array, zip(*sel))
            res[f'{form}_{t}_pred_mean'] = float(pa_.mean())
            res[f'{form}_{t}_abs_err']   = float(np.abs(pa_ - ta_).mean())
            res[f'{form}_{t}_within15']  = float(100*(np.abs(pa_ - ta_) < 15).mean())
        angr = pred_angles(model, rand_test, form)
        res[f'{form}_random_pearson'] = float(pearsonr([a for _,a,_ in angr], [p for _,_,p in angr])[0])
    a = model.encode(list(sts_dev['sentence1']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
    b = model.encode(list(sts_dev['sentence2']), batch_size=256, convert_to_numpy=True, normalize_embeddings=True)
    res['stsb_dev_spearman'] = float(spearmanr((a*b).sum(1), list(sts_dev['score']))[0])
    E = model.encode(texts, batch_size=256, convert_to_numpy=True).astype(np.float64)
    W, *_ = np.linalg.lstsq(E, Y14, rcond=None); P = E @ W
    res['probe_r2_parity'] = float(1 - ((Y14-P)**2).sum() / ((Y14-Y14.mean(0))**2).sum())
    return res

TWINS = [('damp','arid'),('sprint','stroll'),('whisper','shout'),('ascend','plummet'),
 ('inflate','deflate'),('gather','scatter'),('freeze','thaw'),('arrive','depart'),
 ('absorb','emit'),('expand','contract'),('attack','defend'),('borrow','lend'),
 ('buy','sell'),('float','sink'),('melt','solidify'),('sharpen','dull'),
 ('tighten','loosen'),('accelerate','decelerate'),('brighten','darken'),
 ('strengthen','weaken'),('appear','vanish'),('assemble','disassemble'),
 ('encourage','discourage'),('inhale','exhale'),('import','export'),
 ('maximize','minimize'),('ancient','futuristic'),('crowded','deserted'),
 ('fertile','barren'),('flexible','rigid'),('generous','stingy'),('humble','arrogant'),
 ('innocent','guilty'),('optimist','pessimist'),('permanent','temporary'),
 ('scarce','abundant'),('shallow','profound'),('smooth','jagged'),('tame','feral'),
 ('transparent','opaque')]
vocab = name_set | alias_set
tw_keep = [(a,b) for a,b in TWINS if a.upper() not in vocab and b.upper() not in vocab]
print(f'twin probe: {len(tw_keep)} antonym pairs fully OUTSIDE the dictionary')

def twin_angles(model):
    va = model.encode([a for a,b in tw_keep], convert_to_numpy=True, normalize_embeddings=True)
    vb = model.encode([b for a,b in tw_keep], convert_to_numpy=True, normalize_embeddings=True)
    return np.degrees(np.arccos(np.clip((va*vb).sum(1), -1, 1)))

baseline = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
all_res = [eval_model(baseline, 'baseline')]
tw_table = {'pairs': [f'{a}/{b}' for a,b in tw_keep],
            'baseline': [float(x) for x in twin_angles(baseline)]}
del baseline; torch.cuda.empty_cache()
for vname in VARIANTS:
    m = SentenceTransformer(str(MODELS/vname), device='cuda')
    all_res.append(eval_model(m, vname))
    tw_table[vname] = [float(x) for x in twin_angles(m)]
    del m; torch.cuda.empty_cache()

df = pd.DataFrame(all_res).set_index('tag')
pd.set_option('display.width', 220)
cols = ['desc_complement_pred_mean','desc_complement_abs_err','desc_complement_within15',
        'name_complement_abs_err','desc_synonym_abs_err','desc_random_pearson',
        'stsb_dev_spearman','probe_r2_parity']
true_comp_mean = float(np.mean([ta for _,_,t,ta in [(0,0,p[2],p[3]) for p in test_rel] if t=='complement']))
print(f'held-out complement TRUE mean angle: {true_comp_mean:.1f} deg')
print(df[cols].round(3))
df.to_csv(RESULTS/'e1_results.csv')
json.dump(tw_table, open(RESULTS/'e1_twins.json','w'), indent=1)
print('\ntwin probe (unseen antonyms) mean angle:  baseline '
      f"{np.mean(tw_table['baseline']):.1f} deg -> " +
      ',  '.join(f"{v} {np.mean(tw_table[v]):.1f} deg" for v in VARIANTS))

In [ ]:
# ── E1d: pre-registered verdict ───────────────────────────────────────────────
import json, numpy as np, pandas as pd
df = pd.read_csv(RESULTS/'e1_results.csv').set_index('tag')
base = df.loc['baseline']
verdict = {'criteria': 'SUCCESS: held-out complement abs_err < 15 deg AND STS-B drop < 0.03; '
                       'PARTIAL: err reduced >= 50% vs baseline AND drop < 0.08; else FAIL',
           'baseline': {'comp_abs_err': float(base.desc_complement_abs_err),
                        'stsb': float(base.stsb_dev_spearman)}}
best_name, best = None, None
for v in df.index.drop('baseline'):
    r = df.loc[v]
    drop = base.stsb_dev_spearman - r.stsb_dev_spearman
    err  = r.desc_complement_abs_err
    if err < 15 and drop < 0.03: status = 'SUCCESS'
    elif err <= 0.5*base.desc_complement_abs_err and drop < 0.08: status = 'PARTIAL'
    else: status = 'FAIL'
    verdict[v] = {'comp_abs_err': float(err), 'stsb_drop': float(drop), 'status': status,
                  'name_form_err': float(r.name_complement_abs_err),
                  'probe_r2': float(r.probe_r2_parity)}
    print(f'{v:16s} comp_err={err:5.1f} deg  stsb_drop={drop:+.3f}  -> {status}')
    ok = drop < 0.08
    if ok and (best is None or err < best): best, best_name = err, v
verdict['best_variant'] = best_name
json.dump(verdict, open(RESULTS/'verdict.json','w'), indent=1)
print(f'\nBEST VARIANT: {best_name}')
print('\nInterpretation for Movement VI:')
statuses = {verdict[v]['status'] for v in df.index.drop('baseline')}
if 'SUCCESS' in statuses:
    print(' The complement contract CAN be instilled by training — the training-substrate')
    print(' hypothesis reopens in corrected form; the Phase 7 fallback path can generalize.')
elif 'PARTIAL' in statuses:
    print(' Partial instillation: geometry moves toward the contract at real but bounded cost.')
    print(' Report both numbers; the trade-off curve IS the finding.')
else:
    print(' The geometry RESISTS instillation at feasible scale — the 49%-gap claim upgrades')
    print(' from "cannot be projected out" to "cannot be trained in at 22M/384D."')

In [ ]:
# ── Package results (+ browser download) ──────────────────────────────────────
import shutil, json
from pathlib import Path
PKG = Path('/content/geo_pkg'); shutil.rmtree(PKG, ignore_errors=True); PKG.mkdir()
for f in ['e0_layer_sweep.csv','e0_results.json','e0_plot.png',
          'e1_results.csv','e1_twins.json','verdict.json']:
    p = RESULTS/f
    if p.exists(): shutil.copy(p, PKG/f)
best_name = json.load(open(RESULTS/'verdict.json'))['best_variant']
if best_name:
    shutil.copytree(RESULTS/'models'/best_name, PKG/f'model_{best_name}')
zip_path = shutil.make_archive('/content/geo_e0e1_results', 'zip', PKG)
print('packaged:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('manual download: use the Files sidebar ->', zip_path)

In [ ]:
# ── OPTIONAL: also copy results to your Google Drive ─────────────────────────
SAVE_TO_DRIVE = False   # flip to True, then run this cell
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dest = '/content/drive/MyDrive/geo_e0e1_results.zip'
    shutil.copy('/content/geo_e0e1_results.zip', dest)
    print('saved to Drive:', dest)